# ДЗ 2: опыт 1 — 2000 обновлений

**Гипотеза:** более долгое обучение той же модели поможет генерировать тело `solve` и завершать код по EOS. Единственное изменение обучения относительно первого прогона — **200 → 2000 обновлений с нуля**. Seed 42, те же 512 train-пар / 64 validation-пары, UTF-8-байты, Transformer 2+2 слоя ширины 128, batch 8, AdamW `lr=1e-3`, clipping 1.0 и FP32.

В Colab выбери **T4** и выполни все ячейки по порядку в новой сессии. Нужны PyTorch и PyArrow; если отсутствует PyArrow, выполни отдельную ячейку `%pip install pyarrow`. Ноутбук сам скачает один train-файл и test из разрешённого датасета. Обучение начинает со случайных весов; прежний checkpoint не загружается.

После обучения: loss на прежних 16 validation-примерах; loss и greedy-генерации на первых 64 train-парах и всех 64 validation-парах; длины, EOS и AST. Печатаются первые 3 программы каждой части. AST и наличие `solve` — проверки структуры; функциональное качество ими не установлено. Сгенерированные программы не исполняются.

Последняя ячейка измеряет генерацию всех 1919 test-вопросов с прежними batch 32 и cap 512. Таймер включает токенизацию, переносы, генерацию, декодирование и запись predictions; загрузка данных, обучение и диагностика исключены. Остановка проверяется между batch: время может немного превысить 180 с; частичный результат явно помечается.

Скачай **`hw02-byte-2000-results.json`** и сохрани этот ноутбук с outputs. Для следующего опыта сохрани также **`hw02-byte-2000.pt`**. Файлы создаются в текущем каталоге Colab. Метрики записываются после обучения, диагностики и теста; они сохранятся и при остановке до последней ячейки. Имена отличаются от первого прогона.

Связь с лекциями: encoder–decoder, causal mask, teacher forcing и greedy — лекция 2; AdamW и clipping — лекция 3. Число 2000 — наша проверяемая гипотеза. KV-кеш, AMP, BPE и изменение нормировки относятся к отдельным будущим опытам.

Источники: [условие ДЗ](https://github.com/ashaba1in/hse-nlp/blob/7f837940c4af9a686fca7bff5f02345218b6fe80/homeworks/hw2_code_generation.md), [данные](https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/blob/b2724689ff079e9c22efd548fe134bfb29036dbb/README.md), [лекция 2](https://github.com/ashaba1in/hse-nlp/blob/24a73bd327d699039e49e8e169bbce59ac5bbb03/week02_transformer/lecture2.pdf), [лекция 3](https://github.com/ashaba1in/hse-nlp/blob/24a73bd327d699039e49e8e169bbce59ac5bbb03/week03_bert_gpt/lecture3.pdf).

На момент подготовки T4-прогон этого опыта ещё не выполнен. Числа первого прогона ниже — сохранённые результаты пользователя.


In [1]:
# HW2_EXPERIMENT_01: same baseline, 2000 updates from scratch.
import ast
import hashlib
import json
import math
import random
import time
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
import pyarrow.parquet as pq

SEED, STEPS, BATCH_SIZE = 42, 2000, 8
D_MODEL, N_LAYERS, MAX_POSITIONS = 128, 2, 2048
PAD, BOS, EOS, VOCAB = 0, 1, 2, 259  # 256 UTF-8 bytes + 3 special tokens
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Enable GPU in Colab before running this notebook'
device = torch.device('cuda')
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
print('PyTorch:', torch.__version__)


EXPERIMENT = 'byte-2000-steps'
RESULTS_FILE = Path('hw02-byte-2000-results.json')
CHECKPOINT_FILE = 'hw02-byte-2000.pt'
PREDICTIONS_FILE = Path('hw02-byte-2000-predictions.json')
GENERATION_CAP, INFERENCE_BATCH = 512, 32
baseline_saved_run = {
    'source': 'User Colab run, Tesla T4, PyTorch 2.11.0+cu130',
    'steps': 200, 'validation_loss_16': 2.020599365234375,
    'training_seconds': 6.600604578999992, 'completed': 928,
    'test_examples': 1919, 'seconds': 181.070659234,
    'mean_generated_tokens': 512.0, 'complete': False,
}
def save_results():
    RESULTS_FILE.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')


Device: Tesla T4
PyTorch: 2.11.0+cu130


In [2]:
# 1. Data: one allowed shard, a small training subset, and question-level validation.
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
for name, remote in [('train.parquet', 'data/train/part-00000.parquet'), ('test.parquet', 'data/test/test.parquet')]:
    if not (cache / name).exists():
        urlretrieve(f'{base_url}/{remote}', cache / name)
rows = next(pq.ParquetFile(cache / 'train.parquet').iter_batches(batch_size=4096)).to_pylist()
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
short_rows = [r for r in rows if len(r['question'].encode()) <= 512 and len(r['code'].encode()) < 512]
train = [r for r in short_rows if not is_validation(r['question'])][:512]
validation = [r for r in short_rows if is_validation(r['question'])][:64]
assert train and validation
assert not ({r['question'] for r in train} & {r['question'] for r in validation})
test_questions = pq.read_table(cache / 'test.parquet')['question'].to_pylist()
assert len(test_questions) == 1919
assert not ({r['question'] for r in train} & set(test_questions))
print('Rows:', len(train), 'train,', len(validation), 'validation,', len(test_questions), 'test')



Rows: 512 train, 64 validation, 1919 test


In [3]:
# 2. Byte encoding: reversible, no tokenizer training or external vocabulary.
def encode(text):
    return [b + 3 for b in text.encode('utf-8')]
def decode(ids):
    return bytes(i - 3 for i in ids if 3 <= i < VOCAB).decode('utf-8', errors='replace')
def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS]) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c) for c in codes], batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS]) for c in codes], batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels



In [4]:
# 3. Standard encoder-decoder: 2 + 2 layers, learned positions, ReLU, Post-LN.
class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print('Parameters:', sum(p.numel() for p in model.parameters()))



Parameters: 1254915


In [5]:
# 4. Only the number of updates changes; the training loop is unchanged.
if device.type == 'cuda':
    torch.cuda.synchronize()
started = time.perf_counter()
training_log = []
for step in range(1, STEPS + 1):
    model.train()
    src, tgt, labels = training_batch(random.sample(train, BATCH_SIZE))
    logits = model(src, tgt)
    loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step == 1 or step % 50 == 0:
        model.eval()
        with torch.no_grad():
            src, tgt, labels = training_batch(validation[:16])
            val_logits = model(src, tgt)
            val_loss = criterion(val_logits.reshape(-1, VOCAB), labels.reshape(-1)).item()
        entry = {'step': step, 'train_loss': loss.item(), 'validation_loss_16': val_loss}
        training_log.append(entry)
        print('LOSS', json.dumps(entry), flush=True)
if device.type == 'cuda':
    torch.cuda.synchronize()
training_seconds = time.perf_counter() - started
print('Training seconds:', round(training_seconds, 2))
torch.save({'model': {k: v.cpu() for k, v in model.state_dict().items()},
            'steps': STEPS, 'seed': SEED, 'optimizer': optimizer.state_dict()}, CHECKPOINT_FILE)


result = {
    'experiment': EXPERIMENT, 'seed': SEED, 'steps': STEPS,
    'training_examples': len(train), 'validation_examples': len(validation),
    'training_seconds': training_seconds,
    'parameters': sum(p.numel() for p in model.parameters()),
    'torch_version': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
    'dtype': str(next(model.parameters()).dtype), 'tokenizer': 'UTF-8 bytes',
    'config': {'batch_size_train': BATCH_SIZE, 'lr': 1e-3, 'clip_norm': 1.0,
        'encoder_layers': N_LAYERS, 'decoder_layers': N_LAYERS, 'd_model': D_MODEL,
        'heads': 4, 'ffn_dim': 4 * D_MODEL, 'dropout': 0.1, 'norm_first': False,
        'max_positions': MAX_POSITIONS, 'optimizer': 'AdamW',
        'weight_decay': optimizer.param_groups[0]['weight_decay'], 'scheduler': None},
    'data_revision': revision,
    'data_selection': 'first 4096 rows of shard 0; q<=512 bytes, code<512 bytes; seed:question hash split',
    'train_sha256': hashlib.sha256(json.dumps(train, ensure_ascii=False, sort_keys=True).encode()).hexdigest(),
    'validation_sha256': hashlib.sha256(json.dumps(validation, ensure_ascii=False, sort_keys=True).encode()).hexdigest(),
    'training_log': training_log, 'baseline_saved_run': baseline_saved_run,
    'functional_quality_evaluated': False, 'test_status': 'not_started',
}
save_results()
print('Saved:', CHECKPOINT_FILE, str(RESULTS_FILE))


LOSS {"step": 1, "train_loss": 5.694677352905273, "validation_loss_16": 4.621847629547119}
LOSS {"step": 50, "train_loss": 2.2257637977600098, "validation_loss_16": 2.1702208518981934}
LOSS {"step": 100, "train_loss": 2.0023152828216553, "validation_loss_16": 2.079927682876587}
LOSS {"step": 150, "train_loss": 1.9440240859985352, "validation_loss_16": 2.017660617828369}
LOSS {"step": 200, "train_loss": 1.816436529159546, "validation_loss_16": 1.9933820962905884}
LOSS {"step": 250, "train_loss": 1.9635599851608276, "validation_loss_16": 1.9579659700393677}
LOSS {"step": 300, "train_loss": 1.9688537120819092, "validation_loss_16": 1.9070175886154175}
LOSS {"step": 350, "train_loss": 1.9409300088882446, "validation_loss_16": 1.9295880794525146}
LOSS {"step": 400, "train_loss": 1.9714679718017578, "validation_loss_16": 1.8806253671646118}
LOSS {"step": 450, "train_loss": 1.7016887664794922, "validation_loss_16": 1.8650327920913696}
LOSS {"step": 500, "train_loss": 1.831802487373352, "valid

In [6]:
# 5. Fixed train/validation diagnostics. No execution of generated programs.
def inspect_ids(ids):
    active = [i for i in ids if i != PAD]
    code = decode(active)
    try:
        tree = ast.parse(code)
        parses = True
        defines_solve = any(isinstance(node, ast.FunctionDef) and node.name == 'solve'
                            for node in tree.body)
    except (SyntaxError, ValueError):
        parses, defines_solve = False, False
    return {'generated_code': code, 'generated_tokens': len(active),
            'eos_emitted': EOS in active,
            'early_eos': EOS in active and active.index(EOS) + 1 < GENERATION_CAP,
            'reached_cap': len(active) == GENERATION_CAP,
            'parses_as_python': parses, 'defines_solve': defines_solve}

@torch.inference_mode()
def inspect_split(name, rows):
    model.eval()
    loss_sum, label_count = 0.0, 0
    for offset in range(0, len(rows), 16):
        src, tgt, labels = training_batch(rows[offset:offset + 16])
        logits = model(src, tgt)
        loss_sum += nn.functional.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
                                               ignore_index=PAD, reduction='sum').item()
        label_count += labels.ne(PAD).sum().item()
    torch.cuda.synchronize()
    started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        generated = model.generate(source_batch([r['question'] for r in batch]),
                                   max_new_tokens=GENERATION_CAP).cpu().tolist()
        for row, ids in zip(batch, generated):
            examples.append({'question': row['question'], 'reference_code': row['code'],
                             **inspect_ids(ids)})
    torch.cuda.synchronize()
    seconds = time.perf_counter() - started
    lengths = [e['generated_tokens'] for e in examples]
    summary = {'split': name, 'examples': len(rows), 'loss': loss_sum / label_count,
        'mean_generated_tokens': sum(lengths) / len(lengths),
        'min_generated_tokens': min(lengths), 'max_generated_tokens': max(lengths),
        'seconds': seconds, 'answers_per_second': len(rows) / seconds,
        'batch_size_inference': INFERENCE_BATCH, 'max_new_tokens': GENERATION_CAP}
    for metric in ['eos_emitted', 'early_eos', 'reached_cap', 'parses_as_python', 'defines_solve']:
        summary[metric + '_fraction'] = sum(e[metric] for e in examples) / len(examples)
    print('DIAGNOSTICS', json.dumps(summary, ensure_ascii=False), flush=True)
    for example in examples[:3]:
        print('EXAMPLE', name, json.dumps(example, ensure_ascii=False), flush=True)
    return {'summary': summary, 'examples': examples}

# Warm-up is outside diagnostic/test timers and does not update model weights.
model.generate(source_batch([validation[0]['question']]), max_new_tokens=8)
result['diagnostics'] = {}
for name, rows in [('train_64', train[:64]), ('validation_64', validation)]:
    result['diagnostics'][name] = inspect_split(name, rows)
    save_results()
print('Validation loss on the SAME 16 examples:',
      '200 steps =', baseline_saved_run['validation_loss_16'],
      '; 2000 steps =', training_log[-1]['validation_loss_16'])
print('AST / solve checks do not measure functional correctness.')


DIAGNOSTICS {"split": "train_64", "examples": 64, "loss": 1.3539199228447052, "mean_generated_tokens": 421.078125, "min_generated_tokens": 92, "max_generated_tokens": 512, "seconds": 12.683367304, "answers_per_second": 5.045978600636763, "batch_size_inference": 32, "max_new_tokens": 512, "eos_emitted_fraction": 0.34375, "early_eos_fraction": 0.34375, "reached_cap_fraction": 0.65625, "parses_as_python_fraction": 0.046875, "defines_solve_fraction": 0.046875}
EXAMPLE train_64 {"question": "A group of tourists went on a hike. They hiked for 5 hours and took a 30 minute break. If they hiked in total for 10 miles, what was their average speed in miles/hour?", "reference_code": "def solve() -> int:\n    total_time = 5.5 #hours\n    total_distance_travelled = 10 #miles\n    average_speed = total_distance_travelled / total_time\n    result = average_speed\n\n    return result\n", "generated_code": "def solve() -> int:\n    total_tal_t = 30\n            tal_tal_tal_t = + tal_tal_t t t t tal_t t 

In [7]:
# 6. Whole-test timing: same greedy algorithm, batch 32, cap 512, budget 180 s.
# Setup/download/training/diagnostics excluded; tokenization/transfers/save included.
model.eval()
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
answers, generated_tokens = [], 0
eos_count, early_eos_count, cap_count = 0, 0, 0
for offset in range(0, len(test_questions), INFERENCE_BATCH):
    if time.perf_counter() - started >= 180:
        break
    generated = model.generate(source_batch(test_questions[offset:offset + INFERENCE_BATCH]),
                               max_new_tokens=GENERATION_CAP)
    torch.cuda.synchronize()
    for ids in generated.cpu().tolist():
        active = [i for i in ids if i != PAD]
        generated_tokens += len(active)
        eos_count += EOS in active
        early_eos_count += EOS in active and active.index(EOS) + 1 < GENERATION_CAP
        cap_count += len(active) == GENERATION_CAP
        answers.append(decode(ids))
    if len(answers) % 256 == 0:
        print('INFERENCE', len(answers), 'of', len(test_questions), flush=True)
PREDICTIONS_FILE.write_text(json.dumps(answers, ensure_ascii=False), encoding='utf-8')
torch.cuda.synchronize()
seconds = time.perf_counter() - started
test_result = {
    'batch_size_inference': INFERENCE_BATCH, 'max_new_tokens': GENERATION_CAP,
    'test_examples': len(test_questions), 'completed': len(answers), 'seconds': seconds,
    'answers_per_second': len(answers) / seconds, 'generated_tokens': generated_tokens,
    'tokens_per_second': generated_tokens / seconds,
    'mean_generated_tokens': generated_tokens / max(len(answers), 1),
    'eos_emitted_fraction': eos_count / max(len(answers), 1),
    'early_eos_fraction': early_eos_count / max(len(answers), 1),
    'reached_cap_fraction': cap_count / max(len(answers), 1),
    'complete': len(answers) == len(test_questions),
    'within_180_seconds': len(answers) == len(test_questions) and seconds <= 180,
    'peak_gpu_memory_mib': torch.cuda.max_memory_allocated() / 2**20,
    'timer_scope': 'encoding, transfers, greedy generation, CPU decoding, predictions save',
}
result.update(test_result)
result['test_status'] = 'complete' if test_result['complete'] else 'partial'
save_results()
print('TEST', json.dumps(test_result, ensure_ascii=False), flush=True)
print('Saved:', str(RESULTS_FILE), CHECKPOINT_FILE, str(PREDICTIONS_FILE))
print('Download results JSON / notebook with outputs; keep the checkpoint for the next experiment.')


INFERENCE 256 of 1919
INFERENCE 512 of 1919
INFERENCE 768 of 1919
TEST {"batch_size_inference": 32, "max_new_tokens": 512, "test_examples": 1919, "completed": 896, "seconds": 184.11607795600003, "answers_per_second": 4.866495147773708, "generated_tokens": 363320, "tokens_per_second": 1973.320331572705, "mean_generated_tokens": 405.49107142857144, "eos_emitted_fraction": 0.36607142857142855, "early_eos_fraction": 0.36495535714285715, "reached_cap_fraction": 0.6350446428571429, "complete": false, "within_180_seconds": false, "peak_gpu_memory_mib": 604.75341796875, "timer_scope": "encoding, transfers, greedy generation, CPU decoding, predictions save"}
Saved: hw02-byte-2000-results.json hw02-byte-2000.pt hw02-byte-2000-predictions.json
Download results JSON / notebook with outputs; keep the checkpoint for the next experiment.
